In [18]:
import numpy as np
import scanpy as sc
from scipy.sparse import coo_array, save_npz
import os

output_dir = './minimal_dataset/'
def gaussian_penalty(distance, std=100000):
    """Distance penalty for R2G matrix"""
    return 1 - np.exp(-(distance**2) / (2 * std**2))

os.makedirs(output_dir, exist_ok=True)

# Parameters
n_cells = 10
n_genes = 100  # 20 TFs + 80 regular genes
n_tfs = 20
n_regions = 50
max_distance = 1_000_000

# Gene names (TFs first, then regular genes)
gene_names = [f'TF_{i}' for i in range(n_tfs)] + \
            [f'Gene_{i}' for i in range(n_genes - n_tfs)]

# Assign TSS positions to genes (chr1, spread out)
gene_positions = {}
for i, gene in enumerate(gene_names):
    gene_positions[gene] = 50000 + i * 5000  # 5kb spacing

# Region names (chr1:start-end format)
region_names = []
region_positions = {}
for i in range(n_regions):
    start = i * 10000  # 10kb spacing
    end = start + 640
    region_name = f'chr1:{start}-{end}'
    region_names.append(region_name)
    region_positions[region_name] = (start + end) // 2

# 1. Create RNA data (normalized + log-transformed)
print("Creating RNA data...")
X_rna = np.random.lognormal(mean=0, sigma=1, size=(n_cells, n_genes))
X_rna = np.log1p(X_rna)  # Log transform

ad_rna = sc.AnnData(X=X_rna)
ad_rna.var_names = gene_names
ad_rna.obs_names = [f'Cell_{i}' for i in range(n_cells)]
ad_rna.write(output_dir + 'raw_exprMat_train.h5ad')

# 2. Create ATAC data (imputed accessibility - dense, low values)
print("Creating ATAC data...")
X_atac = np.random.beta(a=0.5, b=2, size=(n_cells, n_regions))  # Skewed towards 0

ad_atac = sc.AnnData(X=X_atac)
ad_atac.var_names = region_names
ad_atac.obs_names = [f'Cell_{i}' for i in range(n_cells)]
ad_atac.write(output_dir + 'fragment_matrix_train.h5ad')

# 3. Create R2G matrix (distance-based, realistic)
print("Creating R2G distance matrix...")
rows, cols, values = [], [], []

for region_idx, region_name in enumerate(region_names):
    region_pos = region_positions[region_name]

    for gene_idx, gene_name in enumerate(gene_names):
        gene_tss = gene_positions[gene_name]
        distance = abs(region_pos - gene_tss)

        # Only link if within 1Mb
        if distance <= max_distance:
            penalty = gaussian_penalty(distance, std=100000)
            rows.append(region_idx)
            cols.append(gene_idx)
            values.append(penalty)

r2g = coo_array((values, (rows, cols)), shape=(n_regions, n_genes))
save_npz(output_dir + 'r2g_mask.npz', r2g)

print(f"  R2G matrix: {len(values)} links ({len(values)/(n_regions*n_genes)*100:.1f}% density)")

# 4. Create genome FASTA (chr1 only, 500kb)
print("Creating genome FASTA...")
genome_size = 500000
with open(output_dir + 'genome.fa', 'w') as f:
    f.write('>chr1\n')
    seq = ''.join(np.random.choice(['A','C','G','T'], genome_size))
    # Write in 80-char lines (FASTA format)
    for i in range(0, len(seq), 80):
        f.write(seq[i:i+80] + '\n')

# 5. Create TF list
print("Creating TF list...")
with open(output_dir + 'allTFs_test.txt', 'w') as f:
    for i in range(n_tfs):
        f.write(f'TF_{i}\n')

Creating RNA data...
Creating ATAC data...
Creating R2G distance matrix...
  R2G matrix: 5000 links (100.0% density)
Creating genome FASTA...
Creating TF list...


In [19]:
r2g.toarray()

array([[0.1160945 , 0.13885736, 0.16312881, ..., 0.99999938, 0.99999953,
        0.99999964],
       [0.07570604, 0.09499528, 0.1160945 , ..., 0.99999895, 0.99999919,
        0.99999938],
       [0.04308922, 0.05836271, 0.07570604, ..., 0.99999823, 0.99999863,
        0.99999895],
       ...,
       [0.99985422, 0.99982036, 0.99977917, ..., 0.18874684, 0.21554425,
        0.24335048],
       [0.99990473, 0.999882  , 0.99985422, ..., 0.13885736, 0.16312881,
        0.18874684],
       [0.99993835, 0.99992327, 0.99990473, ..., 0.09499528, 0.1160945 ,
        0.13885736]])